In [6]:
import pandas as pd

# File paths (local)
file_paths = {
    'netflix': '/Users/nityaarya/Downloads/Worth-the-Watch/Data/Raw/netflix_catalog.csv',
    'prime': '/Users/nityaarya/Downloads/Worth-the-Watch/Data/Raw/prime_catalog.csv',
    'hulu': '/Users/nityaarya/Downloads/Worth-the-Watch/Data/Raw/hulu_catalog.csv',
    'hbo_max': '/Users/nityaarya/Downloads/Worth-the-Watch/Data/Raw/hbo_max_catalog.csv',
    'apple_tv': '/Users/nityaarya/Downloads/Worth-the-Watch/Data/Raw/apple_tv_catalog.csv'
}

# Load and clean each catalog
catalogs = {}
for name, path in file_paths.items():
    df = pd.read_csv(path)
    df["platform"] = name  # Add platform column
    df = df.dropna(subset=["title", "type", "genres", "releaseYear", "imdbId"])
    df["releaseYear"] = df["releaseYear"].astype(int)
    df["genres"] = df["genres"].str.split(", ")
    catalogs[name] = df
# Combine all into a single DataFrame
combined_df = pd.concat(catalogs.values(), ignore_index=True)
print(combined_df.head())

combined_df.to_csv("combined_streaming_catalog.csv", index=False)

try:
    combined_path = "/Users/nityaarya/Downloads/Worth-the-Watch/Data/Processed/combined_streaming_catalog.csv"
    combined_df.to_csv(combined_path, index=False)
    print(f"Downloadable file saved to: {combined_path}")
except:
    print("Download path '/Users/nityaarya/Downloads/Worth-the-Watch/Data/Processed/' not available in this environment.")




                 title   type                       genres  releaseYear  \
0                Ariel  movie     [Comedy, Crime, Romance]         1988   
1  Shadows in Paradise  movie       [Comedy, Drama, Music]         1986   
2         Forrest Gump  movie             [Drama, Romance]         1994   
3      American Beauty  movie                      [Drama]         1999   
4    The Fifth Element  movie  [Action, Adventure, Sci-Fi]         1997   

      imdbId  imdbAverageRating  imdbNumVotes availableCountries platform  
0  tt0094675                7.4        9169.0                NaN  netflix  
1  tt0092149                7.4        7994.0                NaN  netflix  
2  tt0109830                8.8     2377322.0                NaN  netflix  
3  tt0169547                8.3     1248325.0                NaN  netflix  
4  tt0119116                7.6      526971.0                NaN  netflix  


In [2]:
import pandas as pd
import requests
import time

TMDB_API_KEY = 'd58935a3624cb12af6521bef5d335d5f'
CSV_PATH = "combined_streaming_catalog.csv"
SAVE_PATH = "combined_streaming_catalog_with_originals.csv"

df = pd.read_csv(CSV_PATH)


def get_tmdb_id(imdb_id):
    """Find TMDb ID and media type from IMDb ID."""
    url = f"https://api.themoviedb.org/3/find/{imdb_id}?external_source=imdb_id&api_key={TMDB_API_KEY}"
    try:
        response = requests.get(url)
        data = response.json()
        for media_type in ['movie_results', 'tv_results']:
            if data.get(media_type):
                return data[media_type][0]['id'], media_type.split('_')[0]
    except Exception as e:
        print(f"[TMDb Lookup Error] {imdb_id}: {e}")
    return None, None

def is_tmdb_original(imdb_id, platform):
    """Check if title is an original for the platform."""
    tmdb_id, media_type = get_tmdb_id(imdb_id)
    if not tmdb_id:
        return "N"

    url = f"https://api.themoviedb.org/3/{media_type}/{tmdb_id}?api_key={TMDB_API_KEY}"
    try:
        response = requests.get(url)
        if response.status_code != 200:
            return "N"
        data = response.json()
    except:
        return "N"

    platform_map = {
        "netflix": "netflix",
        "prime": "amazon",
        "hulu": "hulu",
        "hbo_max": "hbo",
        "apple_tv": "apple"
    }
    target = platform_map.get(platform.lower(), "")

    for company in data.get("production_companies", []) + data.get("networks", []):
        if target in company.get("name", "").lower():
            return "Y"
    return "N"

if "is_original" not in df.columns:
    df["is_original"] = None

for idx, row in df.iterrows():
    if pd.notna(row.get("is_original")) and row["is_original"] in ["Y", "N"]:
        continue

    imdb_id = row.get("imdbId")
    platform = row.get("platform", "")
    print(f"Checking {idx+1}/{len(df)}: {row['title']} ({platform})")

    df.at[idx, "is_original"] = is_tmdb_original(imdb_id, platform)

    # TMDb allows 40 requests per 10 seconds
    if (idx + 1) % 35 == 0:
        print("Sleeping 10 seconds to respect rate limits...")
        time.sleep(10)
    else:
        time.sleep(0.5)


Checking 1/121336: Ariel (netflix)
Checking 2/121336: Shadows in Paradise (netflix)
Checking 3/121336: Forrest Gump (netflix)
Checking 4/121336: American Beauty (netflix)
Checking 5/121336: The Fifth Element (netflix)
Checking 6/121336: Jarhead (netflix)
Checking 7/121336: Unforgiven (netflix)
Checking 8/121336: Eternal Sunshine of the Spotless Mind (netflix)
Checking 9/121336: Amores Perros (netflix)
Checking 10/121336: A History of Violence (netflix)
Checking 11/121336: Talk to Her (netflix)
Checking 12/121336: 8 Mile (netflix)
Checking 13/121336: Paradise Now (netflix)
Checking 14/121336: Million Dollar Baby (netflix)
Checking 15/121336: Billy Elliot (netflix)
Checking 16/121336: American History X (netflix)
Checking 17/121336: Mars Attacks! (netflix)
Checking 18/121336: Before Sunrise (netflix)
Checking 19/121336: Memento (netflix)
Checking 20/121336: Hero (netflix)
Checking 21/121336: Before Sunset (netflix)
Checking 22/121336: Nausicaä of the Valley of the Wind (netflix)
Checking

In [4]:
SAVE_PATH = "/Users/nityaarya/Downloads/Worth-the-Watch/Data/Processed/combined_streaming_catalog_with_originals.csv"
df.to_csv(SAVE_PATH, index=False)
